In [1]:
import pandas as pd
from pathlib import Path

In [2]:
file_path=Path(r"D:\Before-the-Breach\02_DATASET\raw\HIKARI-2021\ALLFLOWMETER_HIKARI2021.csv")

In [3]:
df_sample=pd.read_csv(file_path, nrows=5)

In [4]:
print("COLUMN NAMES:")
print(df_sample.columns.tolist())

COLUMN NAMES:
['Unnamed: 0.1', 'Unnamed: 0', 'uid', 'originh', 'originp', 'responh', 'responp', 'flow_duration', 'fwd_pkts_tot', 'bwd_pkts_tot', 'fwd_data_pkts_tot', 'bwd_data_pkts_tot', 'fwd_pkts_per_sec', 'bwd_pkts_per_sec', 'flow_pkts_per_sec', 'down_up_ratio', 'fwd_header_size_tot', 'fwd_header_size_min', 'fwd_header_size_max', 'bwd_header_size_tot', 'bwd_header_size_min', 'bwd_header_size_max', 'flow_FIN_flag_count', 'flow_SYN_flag_count', 'flow_RST_flag_count', 'fwd_PSH_flag_count', 'bwd_PSH_flag_count', 'flow_ACK_flag_count', 'fwd_URG_flag_count', 'bwd_URG_flag_count', 'flow_CWR_flag_count', 'flow_ECE_flag_count', 'fwd_pkts_payload.min', 'fwd_pkts_payload.max', 'fwd_pkts_payload.tot', 'fwd_pkts_payload.avg', 'fwd_pkts_payload.std', 'bwd_pkts_payload.min', 'bwd_pkts_payload.max', 'bwd_pkts_payload.tot', 'bwd_pkts_payload.avg', 'bwd_pkts_payload.std', 'flow_pkts_payload.min', 'flow_pkts_payload.max', 'flow_pkts_payload.tot', 'flow_pkts_payload.avg', 'flow_pkts_payload.std', 'fwd_i

In [5]:
print("\nFIRST 5 RECORDS:")
display(df_sample)


FIRST 5 RECORDS:


,Unnamed: 0.1,Unnamed: 0,uid,originh,originp,responh,responp,flow_duration,fwd_pkts_tot,bwd_pkts_tot,...,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,traffic_category,Label
0,0,0,Cg61Jch3vdz9DBptj,103.255.15.23,13316,128.199.242.104,443,2.207588,15,14,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,29200,65160,0,Bruteforce-XML,1
1,1,1,CdRIlqLWdj35Y9vW9,103.255.15.23,13318,128.199.242.104,443,15.624266,15,14,...,1.534300e+07,1.534300e+07,1.534300e+07,1.534300e+07,0.0,29200,65160,0,Bruteforce-XML,1
2,2,2,CLzp9Khd0Y09Qkgrg,103.255.15.23,13320,128.199.242.104,443,12.203357,14,13,...,1.196814e+07,1.196814e+07,1.196814e+07,1.196814e+07,0.0,29200,65160,0,Bruteforce-XML,1
3,3,3,Cnf1YA4iLB4CSNWB88,103.255.15.23,13322,128.199.242.104,443,9.992448,14,13,...,9.759205e+06,9.759205e+06,9.759205e+06,9.759205e+06,0.0,29200,65160,0,Bruteforce-XML,1
4,4,4,C4ZKvv3fpO72EAOsJ6,103.255.15.23,13324,128.199.242.104,443,7.780611,14,14,...,7.545305e+06,7.545305e+06,7.545305e+06,7.545305e+06,0.0,29200,65160,0,Bruteforce-XML,1


In [6]:
print("\nNUMBER OF COLUMNS:", len(df_sample.columns))


NUMBER OF COLUMNS: 88


In [7]:
chunksize=100000
total_rows=0
label_counts={}
traffic_counts={}
missing_counts=None

In [8]:
for chunk in pd.read_csv(file_path, chunksize=chunksize, low_memory=False):
    total_rows+=len(chunk)
    if missing_counts is None:
        missing_counts=chunk.isnull().sum()
    else:
        missing_counts+=chunk.isnull().sum()
    if "Label" in chunk.columns:
        counts=chunk["Label"].value_counts(dropna=False)
        for label, count in counts.items():
            label_counts[label]=label_counts.get(label, 0)+count
    if "traffic_category" in chunk.columns:
        counts=chunk["traffic_category"].value_counts(dropna=False)
        for category, count in counts.items():
            traffic_counts[category]=traffic_counts.get(category,0)+count

In [9]:
print("TOTAL RECORXS:", total_rows)

TOTAL RECORXS: 555278


In [10]:
print("\nLABEL DISTRIBUTION:")
print(pd.Series(label_counts))


LABEL DISTRIBUTION:
0    517582
1     37696
dtype: int64


In [11]:
print("\nTRAFFIC CATEGORY DISTRIBUTION:")
print(pd.Series(traffic_counts))


TRAFFIC CATEGORY DISTRIBUTION:
Background             170151
Bruteforce               5884
Bruteforce-XML           5145
Benign                 347431
Probing                 23388
XMRIGCC CryptoMiner      3279
dtype: int64


In [12]:
print("COLUMN WITH MISSING VALUES:")
print(missing_counts[missing_counts>0].sort_values(ascending=False))

COLUMN WITH MISSING VALUES:
Series([], dtype: int64)


In [13]:
print("\nTOTAL MISSING VALUES:", missing_counts.sum())


TOTAL MISSING VALUES: 0


In [14]:
label_category_counts={}

In [15]:
for chunk in pd.read_csv(file_path, chunksize=100000, low_memory=False):
    counts=pd.crosstab(chunk["traffic_category"], chunk["Label"])
    for category in counts.index:
        if category not in label_category_counts:
            label_category_counts[category]={}
        for label in counts.columns:
            label_category_counts[category][label]=(label_category_counts[category].get(label,0)
                +counts.loc[category, label]
                )

In [16]:
print("TRAFFIC CATEGORY vs LABEL:")
print(pd.DataFrame(label_category_counts).T.fillna(0).astype(int))

TRAFFIC CATEGORY vs LABEL:
                          0      1
Background           170151      0
Bruteforce                0   5884
Bruteforce-XML            0   5145
Benign               347431      0
Probing                   0  23388
XMRIGCC CryptoMiner       0   3279


In [17]:
df_sample=pd.read_csv(file_path, nrows=10, low_memory=False)

In [18]:
print("\nDATA TYPES:")
print(df_sample.dtypes.to_string())


DATA TYPES:
Unnamed: 0.1                  int64
Unnamed: 0                    int64
uid                             str
originh                         str
originp                       int64
responh                         str
responp                       int64
flow_duration               float64
fwd_pkts_tot                  int64
bwd_pkts_tot                  int64
fwd_data_pkts_tot             int64
bwd_data_pkts_tot             int64
fwd_pkts_per_sec            float64
bwd_pkts_per_sec            float64
flow_pkts_per_sec           float64
down_up_ratio               float64
fwd_header_size_tot           int64
fwd_header_size_min           int64
fwd_header_size_max           int64
bwd_header_size_tot           int64
bwd_header_size_min           int64
bwd_header_size_max           int64
flow_FIN_flag_count           int64
flow_SYN_flag_count           int64
flow_RST_flag_count           int64
fwd_PSH_flag_count            int64
bwd_PSH_flag_count            int64
flow_ACK_flag_c

In [19]:
print("\nEXTRA COLUMN VALUES:")
print(df_sample[["Unnamed: 0.1", "Unnamed: 0"]].head(10))


EXTRA COLUMN VALUES:
   Unnamed: 0.1  Unnamed: 0
0             0           0
1             1           1
2             2           2
3             3           3
4             4           4
5             5           5
6             6           6
7             7           7
8             8           8
9             9           9


In [20]:
feature_path=r"D:\Before-the-Breach\02_DATASET\raw\NF-CSE-CIC-IDS2018-V3\data\NetFlow_v3_Features.csv"

In [21]:
features=pd.read_csv(feature_path)

In [22]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

In [23]:
display(features)

,Feature,Description
0,IPV4_SRC_ADDR,IPv4 source address
1,IPV4_DST_ADDR,IPv4 destination address
2,L4_SRC_PORT,IPv4 source port number
3,L4_DST_PORT,IPv4 destination port number
4,PROTOCOL,IP protocol identifier byte
5,L7_PROTO,Layer 7 protocol (numeric)
6,IN_BYTES,Incoming number of bytes
7,OUT_BYTES,Outgoing number of bytes
8,IN_PKTS,Incoming number of packets
9,OUT_PKTS,Outgoing number of packets


In [24]:
base_path = Path(r"D:\Before-the-Breach\02_DATASET\raw")

In [25]:
datasets = {
    "UNSW-NB15": base_path / "NF-UNSW-NB15-V3" / "data" / "NF-UNSW-NB15-v3.csv",
    "TON-IoT": base_path / "NF-ToN-IoT-V3" / "data" / "NF-ToN-IoT-v3.csv",
    "BoT-IoT": base_path / "NF-BoT-IoT-V3" / "data" / "NF-BoT-IoT-v3.csv",
    "CSE-CIC-IDS2018": base_path / "NF-CSE-CIC-IDS2018-V3" / "data" / "NF-CICIDS2018-v3.csv"
}

In [26]:
schemas = {}

In [27]:
for name, file_path in datasets.items():
    if file_path.exists():
        columns = pd.read_csv(file_path, nrows=0).columns.tolist()
        schemas[name] = columns
        print(f"\n{name}")
        print("Number of columns:", len(columns))
        print(columns)
    else:
        print(f"\nFile not found: {file_path}")


UNSW-NB15
Number of columns: 55
['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR', 'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'FTP_COMMAND_RET_CODE', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IA

In [28]:
reference_name = "UNSW-NB15"
reference_columns = schemas[reference_name]

In [29]:
for name, columns in schemas.items():
    print(f"\n{'='*60}")
    print(name)
    print("Total columns:", len(columns))
    print("Same column names:", set(columns) == set(reference_columns))
    print("Same column order:", columns == reference_columns)

    missing = set(reference_columns) - set(columns)
    extra = set(columns) - set(reference_columns)

    print("Missing columns:", sorted(missing))
    print("Extra columns:", sorted(extra))

print("\nCOMMON FEATURES ACROSS ALL DATASETS:")
common_columns = set.intersection(
    *(set(columns) for columns in schemas.values())
)


UNSW-NB15
Total columns: 55
Same column names: True
Same column order: True
Missing columns: []
Extra columns: []

TON-IoT
Total columns: 55
Same column names: True
Same column order: True
Missing columns: []
Extra columns: []

BoT-IoT
Total columns: 55
Same column names: True
Same column order: True
Missing columns: []
Extra columns: []

CSE-CIC-IDS2018
Total columns: 55
Same column names: True
Same column order: True
Missing columns: []
Extra columns: []

COMMON FEATURES ACROSS ALL DATASETS:


In [30]:
print("Total common columns:", len(common_columns))
print(sorted(common_columns))

Total common columns: 55
['Attack', 'CLIENT_TCP_FLAGS', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE', 'DNS_TTL_ANSWER', 'DST_TO_SRC_AVG_THROUGHPUT', 'DST_TO_SRC_IAT_AVG', 'DST_TO_SRC_IAT_MAX', 'DST_TO_SRC_IAT_MIN', 'DST_TO_SRC_IAT_STDDEV', 'DST_TO_SRC_SECOND_BYTES', 'DURATION_IN', 'DURATION_OUT', 'FLOW_DURATION_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'FLOW_START_MILLISECONDS', 'FTP_COMMAND_RET_CODE', 'ICMP_IPV4_TYPE', 'ICMP_TYPE', 'IN_BYTES', 'IN_PKTS', 'IPV4_DST_ADDR', 'IPV4_SRC_ADDR', 'L4_DST_PORT', 'L4_SRC_PORT', 'L7_PROTO', 'LONGEST_FLOW_PKT', 'Label', 'MAX_IP_PKT_LEN', 'MAX_TTL', 'MIN_IP_PKT_LEN', 'MIN_TTL', 'NUM_PKTS_1024_TO_1514_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_UP_TO_128_BYTES', 'OUT_BYTES', 'OUT_PKTS', 'PROTOCOL', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SERVER_TCP_FLAGS', 'SHORTEST_FLOW_PKT', 'SRC_TO_DST_AVG_THROUGHPUT', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST

In [31]:
from collections import Counter

In [32]:
for name, file_path in datasets.items():
    label_counts = Counter()
    attack_counts = Counter()
    label_attack_counts = Counter()
    missing_counts = Counter()

    for chunk in pd.read_csv(
        file_path,
        usecols=["Label", "Attack"],
        chunksize=250000,
        low_memory=False
    ):
        label_counts.update(chunk["Label"].value_counts(dropna=False).to_dict())
        attack_counts.update(chunk["Attack"].value_counts(dropna=False).to_dict())

        combinations = chunk.value_counts(
            ["Label", "Attack"],
            dropna=False
        )
        label_attack_counts.update(combinations.to_dict())

        missing_counts.update(chunk.isna().sum().to_dict())


In [34]:
print("\n" + "=" * 65)
print(name)
print("=" * 65)


CSE-CIC-IDS2018


In [35]:
print("\nLABEL DISTRIBUTION:")
print(dict(label_counts))


LABEL DISTRIBUTION:
{0: 17514626, 1: 2600903}


In [37]:
print("\nATTACK CATEGORY DISTRIBUTION:")
print(dict(attack_counts))


ATTACK CATEGORY DISTRIBUTION:
{'Benign': 17514626, 'FTP-BruteForce': 386720, 'SSH-Bruteforce': 188474, 'DoS_attacks-GoldenEye': 61300, 'DoS_attacks-Slowloris': 36040, 'DoS_attacks-SlowHTTPTest': 105550, 'DoS_attacks-Hulk': 100076, 'DDoS_attacks-LOIC-HTTP': 288589, 'DDOS_attack-LOIC-UDP': 3450, 'DDOS_attack-HOIC': 1032311, 'Brute_Force_-Web': 1618, 'Brute_Force_-XSS': 480, 'SQL_Injection': 440, 'Infilteration': 188152, 'Bot': 207703}


In [38]:
print("\nLABEL VS ATTACK:")
print(pd.Series(label_attack_counts).to_string())


LABEL VS ATTACK:
0  Benign                      17514626
1  FTP-BruteForce                386720
   SSH-Bruteforce                188474
   DoS_attacks-GoldenEye          61300
   DoS_attacks-Slowloris          36040
   DoS_attacks-SlowHTTPTest      105550
   DoS_attacks-Hulk              100076
   DDoS_attacks-LOIC-HTTP        288589
   DDOS_attack-LOIC-UDP            3450
   DDOS_attack-HOIC             1032311
   Brute_Force_-Web                1618
   Brute_Force_-XSS                 480
   SQL_Injection                    440
   Infilteration                 188152
   Bot                           207703


In [39]:
print("\nMISSING VALUES:")
print(dict(missing_counts))


MISSING VALUES:
{'Label': 0, 'Attack': 0}


In [40]:
file_path = Path(r"D:\Before-the-Breach\02_DATASET\raw\NF-ToN-IoT-V3\data\NF-ToN-IoT-v3.csv")


In [41]:
label_counts = Counter()
attack_counts = Counter()
label_attack_counts = Counter()
missing_counts = Counter()
total_records = 0

In [42]:
for chunk in pd.read_csv(file_path, usecols=["Label", "Attack"], chunksize=250000, low_memory=False):
    total_records += len(chunk)
    label_counts.update(chunk["Label"].value_counts(dropna=False).to_dict())
    attack_counts.update(chunk["Attack"].value_counts(dropna=False).to_dict())
    label_attack_counts.update(chunk.groupby(["Label", "Attack"], dropna=False).size().to_dict())
    missing_counts.update(chunk.isna().sum().to_dict())

In [44]:
print("=" * 75)
print("DATASET: TON-IoT V3")
print("=" * 75)
print("\nTOTAL RECORDS:")
print(f"{total_records:,}")
print("\nLABEL DISTRIBUTION:")
print(pd.Series(label_counts).sort_index().to_string())
print("\nATTACK CATEGORY DISTRIBUTION:")
print(pd.Series(attack_counts).sort_index().to_string())
print("\nLABEL VS ATTACK CATEGORY:")
print(pd.Series(label_attack_counts).sort_index().to_string())
print("\nMISSING VALUES:")
print(pd.Series(missing_counts).to_string())

DATASET: TON-IoT V3

TOTAL RECORDS:
27,520,260

LABEL DISTRIBUTION:
0    16792214
1    10728046

ATTACK CATEGORY DISTRIBUTION:
Backdoor        203384
Benign        16792214
ddos           4141256
dos             203456
injection       381777
mitm              6013
password       1594777
ransomware        3971
scanning       1358977
xss            2834435

LABEL VS ATTACK CATEGORY:
0  Benign        16792214
1  Backdoor        203384
   ddos           4141256
   dos             203456
   injection       381777
   mitm              6013
   password       1594777
   ransomware        3971
   scanning       1358977
   xss            2834435

MISSING VALUES:
Label     0
Attack    0


In [45]:
file_path = Path(r"D:\Before-the-Breach\02_DATASET\raw\NF-BoT-IoT-V3\data\NF-BoT-IoT-v3.csv")


In [46]:
label_counts = Counter()
attack_counts = Counter()
label_attack_counts = Counter()
missing_counts = Counter()
total_records = 0

In [47]:
for chunk in pd.read_csv(file_path, usecols=["Label", "Attack"], chunksize=250000, low_memory=False):
    total_records += len(chunk)
    label_counts.update(chunk["Label"].value_counts(dropna=False).to_dict())
    attack_counts.update(chunk["Attack"].value_counts(dropna=False).to_dict())
    label_attack_counts.update(chunk.groupby(["Label", "Attack"], dropna=False).size().to_dict())
    missing_counts.update(chunk.isna().sum().to_dict())

In [48]:
print("=" * 75)
print("DATASET: BoT-IoT V3")
print("=" * 75)
print("\nTOTAL RECORDS:")
print(f"{total_records:,}")
print("\nLABEL DISTRIBUTION:")
print(pd.Series(label_counts).sort_index().to_string())
print("\nATTACK CATEGORY DISTRIBUTION:")
print(pd.Series(attack_counts).sort_index().to_string())
print("\nLABEL VS ATTACK CATEGORY:")
print(pd.Series(label_attack_counts).sort_index().to_string())
print("\nMISSING VALUES:")
print(pd.Series(missing_counts).to_string())


DATASET: BoT-IoT V3

TOTAL RECORDS:
16,933,808

LABEL DISTRIBUTION:
0       51989
1    16881819

ATTACK CATEGORY DISTRIBUTION:
Benign              51989
DDoS              7150882
DoS               8034190
Reconnaissance    1695132
Theft                1615

LABEL VS ATTACK CATEGORY:
0  Benign              51989
1  DDoS              7150882
   DoS               8034190
   Reconnaissance    1695132
   Theft                1615

MISSING VALUES:
Label     0
Attack    0


In [49]:
file_path = Path(r"D:\Before-the-Breach\02_DATASET\raw\NF-CSE-CIC-IDS2018-V3\data\NF-CICIDS2018-v3.csv")


In [50]:
label_counts = Counter()
attack_counts = Counter()
label_attack_counts = Counter()
missing_counts = Counter()
total_records = 0

In [51]:
for chunk in pd.read_csv(file_path, usecols=["Label", "Attack"], chunksize=250000, low_memory=False):
    total_records += len(chunk)
    label_counts.update(chunk["Label"].value_counts(dropna=False).to_dict())
    attack_counts.update(chunk["Attack"].value_counts(dropna=False).to_dict())
    label_attack_counts.update(chunk.groupby(["Label", "Attack"], dropna=False).size().to_dict())
    missing_counts.update(chunk.isna().sum().to_dict())

In [52]:
print("=" * 75)
print("DATASET: CSE-CIC-IDS2018 V3")
print("=" * 75)
print("\nTOTAL RECORDS:")
print(f"{total_records:,}")
print("\nLABEL DISTRIBUTION:")
print(pd.Series(label_counts).sort_index().to_string())
print("\nATTACK CATEGORY DISTRIBUTION:")
print(pd.Series(attack_counts).sort_index().to_string())
print("\nLABEL VS ATTACK CATEGORY:")
print(pd.Series(label_attack_counts).sort_index().to_string())
print("\nMISSING VALUES:")
print(pd.Series(missing_counts).to_string())


DATASET: CSE-CIC-IDS2018 V3

TOTAL RECORDS:
20,115,529

LABEL DISTRIBUTION:
0    17514626
1     2600903

ATTACK CATEGORY DISTRIBUTION:
Benign                      17514626
Bot                           207703
Brute_Force_-Web                1618
Brute_Force_-XSS                 480
DDOS_attack-HOIC             1032311
DDOS_attack-LOIC-UDP            3450
DDoS_attacks-LOIC-HTTP        288589
DoS_attacks-GoldenEye          61300
DoS_attacks-Hulk              100076
DoS_attacks-SlowHTTPTest      105550
DoS_attacks-Slowloris          36040
FTP-BruteForce                386720
Infilteration                 188152
SQL_Injection                    440
SSH-Bruteforce                188474

LABEL VS ATTACK CATEGORY:
0  Benign                      17514626
1  Bot                           207703
   Brute_Force_-Web                1618
   Brute_Force_-XSS                 480
   DDOS_attack-HOIC             1032311
   DDOS_attack-LOIC-UDP            3450
   DDoS_attacks-LOIC-HTTP        288589
   